In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Data Engineering\Projects\fleetpulse_project")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print("Project root added:", PROJECT_ROOT)

Project root added: C:\Data Engineering\Projects\fleetpulse_project


In [2]:
from scripts.utils.spark_session import create_spark_session
from scripts.utils.config import(
    BRONZE_BASE_PATH,
    SILVER_BASE_PATH
)

In [3]:
source_table = "branches"
#bronze_path = f"{BRONZE_BASE_PATH}/{table_name}"
silver_path = f"{SILVER_BASE_PATH}/{source_table}"

#print(f"Bronze path: {bronze_path}")
#print("silver_path:", silver_path)

In [4]:
spark = create_spark_session("FleetPulse PySpark Practice")

In [5]:
hconf = spark.sparkContext._jsc.hadoopConfiguration()

print("endpoint:", hconf.get("fs.s3a.endpoint"))
print("access key:", hconf.get("fs.s3a.access.key"))
print("path style:", hconf.get("fs.s3a.path.style.access"))
print("s3a impl:", hconf.get("fs.s3a.impl"))

endpoint: http://localhost:9000
access key: fleetadmin
path style: true
s3a impl: org.apache.hadoop.fs.s3a.S3AFileSystem


In [6]:
source_table = "branches"

bronze_path = f"{BRONZE_BASE_PATH}/{source_table}/"

bronze_branch_df = spark.read.parquet(bronze_path)

bronze_branch_df.select(
    "branch_id",
    "branch_code",
    "branch_name",
    "state",
    "country",
    "branch_status"
).show(truncate=False)

+---------+-----------+------------------------------+-----+-------+-------------+
|branch_id|branch_code|branch_name                   |state|country|branch_status|
+---------+-----------+------------------------------+-----+-------+-------------+
|1        |BR-OH-001  |Youngstown Rental Center      |OH   |USA    |ACTIVE       |
|2        |BR-OH-002  |Cleveland Equipment Hub       |OH   |USA    |ACTIVE       |
|3        |BR-PA-001  |Pittsburgh Industrial Rentals |PA   |USA    |ACTIVE       |
|4        |BR-MI-001  |Detroit Heavy Equipment Center|MI   |USA    |ACTIVE       |
+---------+-----------+------------------------------+-----+-------+-------------+



In [7]:
bronze_branch_df = spark.read.parquet(bronze_path)
silver_branch_df = spark.read.parquet(silver_path)

bronze_branch_df.show()

+---------+-----------+--------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+--------------------+--------------------+----------+--------------------+--------------+------------+----------------+
|branch_id|branch_code|         branch_name|   region|     district|      city|state|country|postal_code| latitude| longitude|branch_status|source_system|          created_at|          updated_at|is_deleted| ingestion_timestamp|ingestion_date|source_table|bronze_load_type|
+---------+-----------+--------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+--------------------+--------------------+----------+--------------------+--------------+------------+----------------+
|        1|  BR-OH-001|Youngstown Rental...|  Midwest|  Ohio Valley|Youngstown|   OH|    USA|      44503|41.099800|-80.649500|       ACTIVE|  RENTAL_OLTP|2026-07-21 21:48:...|202

In [8]:
bronze_branch_df.select(
    "branch_id",
    "branch_code",
    "branch_name",
    "state"
).filter(
    bronze_branch_df.state == "OH"
).show(truncate=False)

+---------+-----------+------------------------+-----+
|branch_id|branch_code|branch_name             |state|
+---------+-----------+------------------------+-----+
|1        |BR-OH-001  |Youngstown Rental Center|OH   |
|2        |BR-OH-002  |Cleveland Equipment Hub |OH   |
+---------+-----------+------------------------+-----+



In [10]:
from pyspark.sql.functions import col

In [11]:
bronze_branch_df.select(
    "branch_id",
    "branch_code",
    "branch_name",
    "state"
).filter(col("state") == "OH").show(truncate=False)

+---------+-----------+------------------------+-----+
|branch_id|branch_code|branch_name             |state|
+---------+-----------+------------------------+-----+
|1        |BR-OH-001  |Youngstown Rental Center|OH   |
|2        |BR-OH-002  |Cleveland Equipment Hub |OH   |
+---------+-----------+------------------------+-----+



In [12]:
bronze_branch_df.groupBy("state").count().show()

+-----+-----+
|state|count|
+-----+-----+
|   MI|    1|
|   OH|    2|
|   PA|    1|
+-----+-----+



In [13]:
from pyspark.sql.functions import count

bronze_branch_df.groupBy("state").agg(
    count("*").alias("branch_count")
).show()

+-----+------------+
|state|branch_count|
+-----+------------+
|   MI|           1|
|   OH|           2|
|   PA|           1|
+-----+------------+



In [22]:
practice_data = [
    (1, "BR-OH-001", "Youngstown Rental Center", "Midwest", "2026-07-21 10:00:00"),
    (1, "BR-OH-001", "Youngstown Rental Center Updated", "Great Lakes", "2026-07-22 10:00:00"),
    (2, "BR-OH-002", "Cleveland Equipment Hub", "Midwest", "2026-07-21 11:00:00"),
    (2, "BR-OH-002", "Cleveland Equipment Hub", "Midwest", "2026-07-21 11:00:00"),
    (3, "BR-PA-001", "Pittsburgh Industrial Rentals", "Northeast", "2026-07-21 12:00:00"),
]

practice_columns = [
    "branch_id",
    "branch_code",
    "branch_name",
    "region",
    "updated_at"
]

practice_df = spark.createDataFrame(practice_data, practice_columns)
practice_df.show(truncate=False)

+---------+-----------+--------------------------------+-----------+-------------------+
|branch_id|branch_code|branch_name                     |region     |updated_at         |
+---------+-----------+--------------------------------+-----------+-------------------+
|1        |BR-OH-001  |Youngstown Rental Center        |Midwest    |2026-07-21 10:00:00|
|1        |BR-OH-001  |Youngstown Rental Center Updated|Great Lakes|2026-07-22 10:00:00|
|2        |BR-OH-002  |Cleveland Equipment Hub         |Midwest    |2026-07-21 11:00:00|
|2        |BR-OH-002  |Cleveland Equipment Hub         |Midwest    |2026-07-21 11:00:00|
|3        |BR-PA-001  |Pittsburgh Industrial Rentals   |Northeast  |2026-07-21 12:00:00|
+---------+-----------+--------------------------------+-----------+-------------------+



In [28]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("branch_id").orderBy(col("updated_at").desc())

In [30]:
practice_ranked_df = practice_df.withColumn(
    "row_num", row_number().over(window_spec)
)

practice_ranked_df.show(truncate=False)

+---------+-----------+--------------------------------+-----------+-------------------+-------+
|branch_id|branch_code|branch_name                     |region     |updated_at         |row_num|
+---------+-----------+--------------------------------+-----------+-------------------+-------+
|1        |BR-OH-001  |Youngstown Rental Center Updated|Great Lakes|2026-07-22 10:00:00|1      |
|1        |BR-OH-001  |Youngstown Rental Center        |Midwest    |2026-07-21 10:00:00|2      |
|2        |BR-OH-002  |Cleveland Equipment Hub         |Midwest    |2026-07-21 11:00:00|1      |
|2        |BR-OH-002  |Cleveland Equipment Hub         |Midwest    |2026-07-21 11:00:00|2      |
|3        |BR-PA-001  |Pittsburgh Industrial Rentals   |Northeast  |2026-07-21 12:00:00|1      |
+---------+-----------+--------------------------------+-----------+-------------------+-------+



In [33]:
practice_last_df = practice_ranked_df.filter(
    col("row_num") == 1
)
practice_last_df.show(truncate=False)

+---------+-----------+--------------------------------+-----------+-------------------+-------+
|branch_id|branch_code|branch_name                     |region     |updated_at         |row_num|
+---------+-----------+--------------------------------+-----------+-------------------+-------+
|1        |BR-OH-001  |Youngstown Rental Center Updated|Great Lakes|2026-07-22 10:00:00|1      |
|2        |BR-OH-002  |Cleveland Equipment Hub         |Midwest    |2026-07-21 11:00:00|1      |
|3        |BR-PA-001  |Pittsburgh Industrial Rentals   |Northeast  |2026-07-21 12:00:00|1      |
+---------+-----------+--------------------------------+-----------+-------------------+-------+



In [34]:
table = "customers"
path = f"{BRONZE_BASE_PATH}/{table}"
customer_df = spark.read.parquet(path)

In [35]:
customer_df.show(truncate=False)

+-----------+---------------+------------------------+-------------+-------------------+-------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|customer_id|customer_number|customer_name           |customer_type|industry           |contact_email                  |contact_phone|billing_city|billing_state|billing_country|credit_status|customer_status|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+-----------+---------------+------------------------+-------------+-------------------+-------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+-------------------------+-------------------------+----------+--

In [36]:
bronze_branch_df.show(truncate=False)

+---------+-----------+------------------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|branch_id|branch_code|branch_name                   |region   |district     |city      |state|country|postal_code|latitude |longitude |branch_status|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+---------+-----------+------------------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|1        |BR-OH-001  |Youngstown Rental Center      |Midwest  |Ohio Valley  |Youngstown|OH   |USA  

In [38]:
selected_branch_columns = bronze_branch_df.select(
    "branch_id",
    "branch_name",
    "region",
    "state",
    "branch_status"
)

selected_branch_columns.show()

+---------+--------------------+---------+-----+-------------+
|branch_id|         branch_name|   region|state|branch_status|
+---------+--------------------+---------+-----+-------------+
|        1|Youngstown Rental...|  Midwest|   OH|       ACTIVE|
|        2|Cleveland Equipme...|  Midwest|   OH|       ACTIVE|
|        3|Pittsburgh Indust...|Northeast|   PA|       ACTIVE|
|        4|Detroit Heavy Equ...|  Midwest|   MI|       ACTIVE|
+---------+--------------------+---------+-----+-------------+



In [51]:
import pyspark.sql.functions as F

selected_branch_columns.groupBy("region", "state").agg(
    F.count("*").alias("branch_count"),
    F.sum(F.when(F.col("branch_status") == "ACTIVE", 1).otherwise(0)).alias("active_branch_count")
).orderBy("region", "state").show(truncate=False)

+---------+-----+------------+-------------------+
|region   |state|branch_count|active_branch_count|
+---------+-----+------------+-------------------+
|Midwest  |MI   |1           |1                  |
|Midwest  |OH   |2           |2                  |
|Northeast|PA   |1           |1                  |
+---------+-----+------------+-------------------+



In [52]:
customer_df.show(truncate=False)

+-----------+---------------+------------------------+-------------+-------------------+-------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|customer_id|customer_number|customer_name           |customer_type|industry           |contact_email                  |contact_phone|billing_city|billing_state|billing_country|credit_status|customer_status|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+-----------+---------------+------------------------+-------------+-------------------+-------------------------------+-------------+------------+-------------+---------------+-------------+---------------+-------------+-------------------------+-------------------------+----------+--

In [ ]:
result_df = customer_df.groupBy("customer_type", "credit_status").agg(
    F.sum(F.when(F.col("customer_status") == "ACTIVE", 1).otherwise(0)).alias("customer_count")
)
result_df.show(truncate=False)


+-------------+-------------+--------------+
|customer_type|credit_status|customer_count|
+-------------+-------------+--------------+
|COMMERCIAL   |HOLD         |1             |
|COMMERCIAL   |GOOD         |2             |
|GOVERNMENT   |GOOD         |1             |
|COMMERCIAL   |REVIEW       |1             |
+-------------+-------------+--------------+

+-------------+-------------+--------------+
|customer_type|credit_status|customer_count|
+-------------+-------------+--------------+
|COMMERCIAL   |HOLD         |1             |
|COMMERCIAL   |GOOD         |2             |
|GOVERNMENT   |GOOD         |1             |
|COMMERCIAL   |REVIEW       |1             |
+-------------+-------------+--------------+



In [60]:
table = "equipment"
path = f"{BRONZE_BASE_PATH}/{table}"
equipment_df = spark.read.parquet(path)
equipment_df.show(truncate=False)

+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+------------+----------------+
|equipment_id|equipment_number|serial_number|category_id|owning_branch_id|current_branch_id|manufacturer|model  |model_year|acquisition_date|acquisition_cost|equipment_status|telematics_enabled|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table|bronze_load_type|
+------------+----------------+-------------+-----------+----------------+-----------------+------------+-------+----------+----------------+----------------+----------------+------------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------

In [67]:
equipment_window_spc = Window.partitionBy("equipment_id").orderBy(col("updated_at").desc())
equipment_status_history_df = equipment_df.withColumn("row_num", row_number().over(equipment_window_spc))
equipment_status_history_df.select(
    "equipment_id",
    "equipment_status",
    "created_at",
    "updated_at",
    "row_num"
).show(truncate=False)

+------------+----------------+-------------------------+-------------------------+-------+
|equipment_id|equipment_status|created_at               |updated_at               |row_num|
+------------+----------------+-------------------------+-------------------------+-------+
|1           |RENTED          |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|1      |
|2           |AVAILABLE       |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|1      |
|3           |RENTED          |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|1      |
|4           |MAINTENANCE     |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|1      |
|5           |AVAILABLE       |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|1      |
|6           |RENTED          |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|1      |
+------------+----------------+-------------------------+-------------------------+-------+

